In [2]:
import pandas as pd
df = pd.read_csv("hr_clean.csv")
df.shape

(1470, 32)

In [3]:
Y = df["AttritionFlag"]
X = df.drop(columns=["Attrition", "AttritionFlag"])
X = pd.get_dummies(X, drop_first=True)

In [4]:
X = pd.get_dummies(X, drop_first=True)

X.shape

(1470, 44)

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(X_train.shape, X_test.shape)

(1176, 44) (294, 44)


In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000, class_weight="balanced")

log_model.fit(X_train_scaled, Y_train)
log_pred = log_model.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(Y_test, log_pred))
print(classification_report(Y_test, log_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

rf_model.fit(X_train, Y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest (default 0.5 threshold)")
print(confusion_matrix(Y_test, rf_pred))
print(classification_report(Y_test, rf_pred))

rf_proba = rf_model.predict_proba(X_test)[:, 1]

rf_pred_30 = (rf_proba >= 0.3).astype(int)

print("Random Forest (0.3 threshold)")
print(confusion_matrix(Y_test, rf_pred_30))
print(classification_report(Y_test, rf_pred_30))

In [ ]:
coefs = pd.Series(log_model.coef_[0], index=X.columns).sort_values()

print("Top 5 Risk Factors:")
print(coefs.tail(5))

print("\nTop 5 Protective Factors:")
print(coefs.head(5))

In [ ]:
import matplotlib.pyplot as plt

coefs.tail(10).plot(kind="barh")
plt.title("Top 10 Employee Attrition Risk Factors")
plt.show()

In [19]:
import joblib

joblib.dump(log_model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(list(X.columns), "columns.pkl")


['columns.pkl']